# Instalment Payments Data Cleaning

This notebook cleans the instalment-payment records using the problems found during EDA. It keeps records linked to the project applicants, checks for repeated payment entries, flags late payments and underpayments, and removes features with too many missing values.


## Import libraries


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
print("Libraries imported successfully.")

Libraries imported successfully.


## Set project paths


In [5]:
current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
raw_path = project_root / "data" / "raw" / "installments_payments.csv"
training_ids_path = project_root / "data" / "modeling" / "splits" / "training_ids.csv"
test_ids_path = project_root / "data" / "modeling" / "splits" / "test_ids.csv"
output_path = project_root / "data" / "interim" / "installments_payments_clean.pkl"
audit_folder = project_root / "reports" / "audits"
output_path.parent.mkdir(parents=True, exist_ok=True)
audit_folder.mkdir(parents=True, exist_ok=True)
for required_path in [raw_path, training_ids_path, test_ids_path]:
    assert required_path.exists(), f"Required file was not found: {required_path}"
print("Raw input:", raw_path)
print("Clean output:", output_path)

Raw input: /Users/taranveersingh/A-MRP/data/raw/installments_payments.csv
Clean output: /Users/taranveersingh/A-MRP/data/interim/installments_payments_clean.pkl


## Load data and retain project applicants


In [7]:
installments_raw = pd.read_csv(raw_path)
training_ids = pd.read_csv(training_ids_path)["SK_ID_CURR"]
test_ids = pd.read_csv(test_ids_path)["SK_ID_CURR"]
training_id_set = set(training_ids)
project_ids = training_id_set.union(set(test_ids))
original_rows, original_columns = installments_raw.shape

installments_clean = installments_raw.loc[
    installments_raw["SK_ID_CURR"].isin(project_ids)
].copy().reset_index(drop=True)
del installments_raw
out_of_scope_rows = original_rows - len(installments_clean)
print("Raw rows:", original_rows)
print("Project rows retained:", len(installments_clean))
print("Out-of-scope rows removed:", out_of_scope_rows)
print("Applicants with installment history:", installments_clean["SK_ID_CURR"].nunique())

Raw rows: 13605401
Project rows retained: 11591592
Out-of-scope rows removed: 2013809
Applicants with installment history: 291643


The rows that were removed belong to applicants outside the project's training and test data.


## Validate identifiers and duplicates


In [10]:
missing_current_ids = int(installments_clean["SK_ID_CURR"].isna().sum())
missing_previous_ids = int(installments_clean["SK_ID_PREV"].isna().sum())
exact_duplicates = int(installments_clean.duplicated().sum())
repeated_installment_rows = int(installments_clean.duplicated(
    ["SK_ID_PREV", "NUM_INSTALMENT_NUMBER", "NUM_INSTALMENT_VERSION"], keep=False
).sum())
if exact_duplicates > 0:
    installments_clean = installments_clean.drop_duplicates().reset_index(drop=True)
assert missing_current_ids == 0 and missing_previous_ids == 0
print("Missing applicant IDs:", missing_current_ids)
print("Missing previous-loan IDs:", missing_previous_ids)
print("Exact duplicate rows removed:", exact_duplicates)
print("Rows belonging to repeated installment keys (retained):", repeated_installment_rows)

Missing applicant IDs: 0
Missing previous-loan IDs: 0
Exact duplicate rows removed: 0
Rows belonging to repeated installment keys (retained): 1118890


No missing IDs or exact duplicate rows were found. The rows that share the same instalment key (loan, version, number) are kept, since they represent payments made in several parts, not accidental duplicates.


## Audit payment dates and amounts


In [13]:
future_due_date = installments_clean["DAYS_INSTALMENT"].gt(0)
future_payment_date = installments_clean["DAYS_ENTRY_PAYMENT"].gt(0)
negative_installment = installments_clean["AMT_INSTALMENT"].lt(0)
negative_payment = installments_clean["AMT_PAYMENT"].lt(0)
zero_installment = installments_clean["AMT_INSTALMENT"].eq(0)
zero_payment = installments_clean["AMT_PAYMENT"].eq(0)
late_payment = installments_clean["DAYS_ENTRY_PAYMENT"].gt(installments_clean["DAYS_INSTALMENT"])
underpayment = installments_clean["AMT_PAYMENT"].lt(installments_clean["AMT_INSTALMENT"])
overpayment = installments_clean["AMT_PAYMENT"].gt(installments_clean["AMT_INSTALMENT"])

installments_clean["INST_DATE_ANOMALY"] = (future_due_date | future_payment_date).astype("int8")
installments_clean["INST_AMOUNT_ANOMALY"] = (negative_installment | negative_payment).astype("int8")
installments_clean["INST_ZERO_PAYMENT"] = zero_payment.astype("int8")
installments_clean["INST_LATE_PAYMENT"] = late_payment.fillna(False).astype("int8")
installments_clean["INST_UNDERPAYMENT"] = underpayment.fillna(False).astype("int8")
installments_clean["INST_OVERPAYMENT"] = overpayment.fillna(False).astype("int8")
installments_clean.loc[future_due_date, "DAYS_INSTALMENT"] = np.nan
installments_clean.loc[future_payment_date, "DAYS_ENTRY_PAYMENT"] = np.nan
installments_clean.loc[negative_installment, "AMT_INSTALMENT"] = np.nan
installments_clean.loc[negative_payment, "AMT_PAYMENT"] = np.nan
print("Future due dates corrected:", int(future_due_date.sum()))
print("Future payment dates corrected:", int(future_payment_date.sum()))
print("Negative amount values corrected:", int((negative_installment | negative_payment).sum()))
print("Zero scheduled installments retained:", int(zero_installment.sum()))
print("Zero payments retained and flagged:", int(zero_payment.sum()))
print("Late-payment records:", int(late_payment.sum()))
print("Underpayment records:", int(underpayment.sum()))
print("Overpayment records:", int(overpayment.sum()))

Future due dates corrected: 0
Future payment dates corrected: 0
Negative amount values corrected: 0
Zero scheduled installments retained: 269
Zero payments retained and flagged: 1259
Late-payment records: 993629
Underpayment records: 1119901
Overpayment records: 146780


No future dates or negative amounts were found. A small number of zero-amount records are kept and flagged, since they could still be genuine. Underpayments turn out to be more common here than late payments, which lines up with what was seen during EDA.


## Build training-only feature decisions


In [16]:
MISSING_THRESHOLD = 0.50
training_installments = installments_clean.loc[
    installments_clean["SK_ID_CURR"].isin(training_id_set)
]
decision_rows = []
for column in installments_clean.columns:
    if column in ["SK_ID_CURR", "SK_ID_PREV"]:
        continue
    series = training_installments[column]
    missing_rate = series.isna().mean()
    unique_non_missing = series.nunique(dropna=True)
    decision = "Keep"
    reason = "Retain for applicant-level aggregation and later target-based selection"
    if missing_rate >= MISSING_THRESHOLD:
        decision = "Remove"
        reason = f"Training-linked missing rate is at least {MISSING_THRESHOLD:.0%}"
    elif unique_non_missing <= 1:
        decision = "Remove"
        reason = "Constant among training-linked records"
    decision_rows.append({
        "feature": column, "data_type": str(series.dtype),
        "missing_count": int(series.isna().sum()), "missing_rate": missing_rate,
        "unique_non_missing": int(unique_non_missing), "decision": decision,
        "reason": reason, "target_association_stage": "After applicant-level aggregation"
    })
feature_decisions = pd.DataFrame(decision_rows).sort_values(
    ["decision", "missing_rate"], ascending=[True, False]
).reset_index(drop=True)
removed_features = feature_decisions.loc[
    feature_decisions["decision"] == "Remove", "feature"
] .tolist()
installments_clean = installments_clean.drop(columns=removed_features)
print("Features removed:", removed_features)
feature_decisions.round(5)

Features removed: ['INST_DATE_ANOMALY', 'INST_AMOUNT_ANOMALY']


,feature,data_type,missing_count,missing_rate,unique_non_missing,decision,reason,target_association_stage
0,DAYS_ENTRY_PAYMENT,float64,2046,0.00022,3035,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
1,AMT_PAYMENT,float64,2046,0.00022,789553,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
2,NUM_INSTALMENT_VERSION,float64,0,0.00000,55,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
3,NUM_INSTALMENT_NUMBER,int64,0,0.00000,277,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
4,DAYS_INSTALMENT,float64,0,0.00000,2922,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
5,AMT_INSTALMENT,float64,0,0.00000,744519,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
6,INST_ZERO_PAYMENT,int8,0,0.00000,2,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
7,INST_LATE_PAYMENT,int8,0,0.00000,2,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
8,INST_UNDERPAYMENT,int8,0,0.00000,2,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
9,INST_OVERPAYMENT,int8,0,0.00000,2,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation


Only two features were removed: the date and amount anomaly flags created earlier in this notebook. Since no future dates or negative amounts were actually found, both of these ended up constant and were dropped.


## Create record-level missingness features


In [19]:
record_features = [c for c in installments_clean.columns if c not in ["SK_ID_CURR", "SK_ID_PREV"]]
installments_clean["INST_RECORD_MISSING_COUNT"] = installments_clean[record_features].isna().sum(axis=1).astype("int8")
installments_clean["INST_RECORD_MISSING_RATE"] = installments_clean["INST_RECORD_MISSING_COUNT"] / len(record_features)
print(installments_clean[["INST_RECORD_MISSING_COUNT", "INST_RECORD_MISSING_RATE"]].describe().round(5))

       INST_RECORD_MISSING_COUNT  INST_RECORD_MISSING_RATE
count               1.159159e+07              1.159159e+07
mean                4.500000e-04              4.000000e-05
std                 2.985000e-02              2.990000e-03
min                 0.000000e+00              0.000000e+00
25%                 0.000000e+00              0.000000e+00
50%                 0.000000e+00              0.000000e+00
75%                 0.000000e+00              0.000000e+00
max                 2.000000e+00              2.000000e-01


These two columns track how much information is missing for each instalment record. The missing rate is very low here, well under 1%, since only DAYS_ENTRY_PAYMENT and AMT_PAYMENT have any missing values.


## Validate the cleaned table


In [22]:
numeric_columns = installments_clean.select_dtypes(include="number").columns
infinite_count = sum(int(np.isinf(installments_clean[c].dropna()).sum()) for c in numeric_columns)
validation_checks = pd.DataFrame([
    {"check": "Only project applicants retained", "passed": set(installments_clean["SK_ID_CURR"]).issubset(project_ids)},
    {"check": "Applicant IDs complete", "passed": installments_clean["SK_ID_CURR"].notna().all()},
    {"check": "Previous-loan IDs complete", "passed": installments_clean["SK_ID_PREV"].notna().all()},
    {"check": "No exact duplicates", "passed": not installments_clean.duplicated().any()},
    {"check": "No future installment dates", "passed": not installments_clean["DAYS_INSTALMENT"].gt(0).any()},
    {"check": "No future payment dates", "passed": not installments_clean["DAYS_ENTRY_PAYMENT"].gt(0).any()},
    {"check": "No negative installment amounts", "passed": not installments_clean["AMT_INSTALMENT"].lt(0).any()},
    {"check": "No negative payment amounts", "passed": not installments_clean["AMT_PAYMENT"].lt(0).any()},
    {"check": "No high-missing retained feature", "passed": not (feature_decisions.query("decision == 'Keep'")["missing_rate"] >= MISSING_THRESHOLD).any()},
    {"check": "No infinite numerical values", "passed": infinite_count == 0},
])
assert validation_checks["passed"].all(), "At least one installment cleaning check failed."
validation_checks

,check,passed
0,Only project applicants retained,True
1,Applicant IDs complete,True
2,Previous-loan IDs complete,True
3,No exact duplicates,True
4,No future installment dates,True
5,No future payment dates,True
6,No negative installment amounts,True
7,No negative payment amounts,True
8,No high-missing retained feature,True
9,No infinite numerical values,True


All checks passed.


## Save the clean table and audit reports


In [25]:
cleaning_audit = pd.DataFrame([
    {"rule": "Out-of-scope records removed", "affected": out_of_scope_rows},
    {"rule": "Exact duplicate rows removed", "affected": exact_duplicates},
    {"rule": "Repeated installment-key rows retained", "affected": repeated_installment_rows},
    {"rule": "Late-payment records flagged", "affected": int(late_payment.sum())},
    {"rule": "Underpayment records flagged", "affected": int(underpayment.sum())},
    {"rule": "Overpayment records flagged", "affected": int(overpayment.sum())},
    {"rule": "Features removed by missingness/constant policy", "affected": len(removed_features)},
])
installments_clean.to_pickle(output_path)
feature_decisions.to_csv(audit_folder / "installments_feature_decisions.csv", index=False)
cleaning_audit.to_csv(audit_folder / "installments_cleaning_audit.csv", index=False)
validation_checks.to_csv(audit_folder / "installments_cleaning_validation.csv", index=False)
print("Clean installments dataset saved:", output_path)
print("Output rows:", len(installments_clean))
print("Output columns:", installments_clean.shape[1])
print("Unique applicants:", installments_clean["SK_ID_CURR"].nunique())
print("Remaining numerical missing values:", int(installments_clean.select_dtypes(include="number").isna().sum().sum()))

Clean installments dataset saved: /Users/taranveersingh/A-MRP/data/interim/installments_payments_clean.pkl
Output rows: 11591592
Output columns: 14
Unique applicants: 291643
Remaining numerical missing values: 5166


## Main cleaning results

The cleaned instalment-payments data contains 11,591,592 records for 291,643 applicants. No missing or duplicate IDs remain.

Repeated payment rows for the same instalment were kept, since they represent payments made in several parts. Late payments, underpayments, and overpayments were flagged rather than removed.

Only two features were removed, both because they ended up constant. The cleaned data has 14 columns, with very few numerical missing values remaining. The next step is to clean the bureau-balance data.
